# Fraud detection ML pipeline (CRISP-DM)

**Course-style notebook:** predict `is_fraud` for e-commerce orders using SQLite `shop.db`.

This notebook is organized to mirror the CRISP-DM lifecycle: business understanding → data understanding → preparation → modeling → evaluation → deployment.


## 1. Business Understanding

### Problem in plain English
When an online shop takes an order, some transactions are **fraudulent** (stolen cards, account takeover, etc.). The business wants a model that **flags risky orders early** so they can be reviewed or blocked before goods ship and money is lost.

### Business goal
**Predict whether an order is fraudulent** (`is_fraud = 1`) using information that is realistically known **at checkout time** (customer profile, cart, payment channel, device, location signals, etc.).

### Success criteria (why more than accuracy?)
Fraud is usually **rare**, so **accuracy can look high even if the model never catches fraud**. We care about:

| Metric | What it measures | Why it matters for fraud |
|--------|------------------|---------------------------|
| **Precision** | Of predicted frauds, how many were real? | High precision → fewer false alarms; review teams are not overwhelmed. |
| **Recall** | Of true frauds, how many did we catch? | High recall → fewer fraudulent orders slip through (lower losses). |
| **F1** | Harmonic mean of precision and recall | Single score when you need balance. |
| **ROC AUC** | Ability to rank fraud above non-fraud across thresholds | Useful for comparing models and tuning operating points. |

**Precision vs recall tradeoff:** Catching more fraud usually means **more false positives** (legitimate orders flagged). The business must choose an operating threshold: banks often favor **recall** (missed fraud is expensive), while customer experience teams push for **precision** (too many blocks anger real customers). There is no single “best” answer—stakeholders set the balance.


## 2. Data Understanding

We connect to `shop.db`, list tables, and profile the `orders` table plus related tables that are safe to use **at order time** (we **exclude** post-order outcomes such as shipment labels when building features for fraud at checkout—see notes in code).


In [ ]:
import sqlite3
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

# Resolve database path from current working directory (project root expected)
ROOT = Path.cwd().resolve()
DB_PATH = ROOT / "shop.db"
assert DB_PATH.exists(), f"Could not find shop.db at {DB_PATH}. Run notebook from project root."

print("Using database:", DB_PATH)


In [ ]:
conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql(
    """
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """,
    conn,
)
print("Tables and views:")
print(tables.to_string(index=False))

# Detailed column info for user tables
for t in tables.loc[tables["type"] == "table", "name"]:
    info = pd.read_sql(f'PRAGMA table_info("{t}")', conn)
    print(f"\n--- {t} ---")
    print(info[["name", "type"]].to_string(index=False))


In [ ]:
orders_head = pd.read_sql("SELECT * FROM orders LIMIT 5", conn)
print("orders sample:")
try:
    from IPython.display import display
except ImportError:
    display = print
display(orders_head)


In [ ]:
# Class balance for the target
balance = pd.read_sql(
    "SELECT is_fraud, COUNT(*) AS n FROM orders GROUP BY is_fraud ORDER BY is_fraud",
    conn,
)
print(balance)
fraud_rate = balance.loc[balance["is_fraud"] == 1, "n"].sum() / balance["n"].sum()
print(f"\nFraud rate: {fraud_rate:.3%}")


In [ ]:
# Load core tables into pandas for EDA (full orders for profiling)
orders_df = pd.read_sql("SELECT * FROM orders", conn)
print("orders shape:", orders_df.shape)
print("\nDtypes:\n", orders_df.dtypes)
print("\nMissing values (orders):\n", orders_df.isna().sum().sort_values(ascending=False).head(20))


In [ ]:
# Related tables (useful at order time)
customers_df = pd.read_sql("SELECT * FROM customers", conn)
items_df = pd.read_sql("SELECT * FROM order_items", conn)
products_df = pd.read_sql("SELECT * FROM products", conn)

print("customers", customers_df.shape)
print("order_items", items_df.shape)
print("products", products_df.shape)

# Shipments exist but ship AFTER the order; we do not use them as features for checkout-time fraud.
ship_chk = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' AND name='shipments'",
    conn,
)
if len(ship_chk):
    print("\nNote: `shipments` table exists - excluded from features (post-order outcome).")


In [ ]:
# EDA: summary stats on numeric columns in orders
num_cols_orders = orders_df.select_dtypes(include=[np.number]).columns.tolist()
summary = orders_df[num_cols_orders].describe().T
print(summary)


In [ ]:
# Distributions for key numeric features (orders)
plot_cols = [c for c in ["order_total", "order_subtotal", "shipping_fee", "tax_amount", "risk_score"] if c in orders_df.columns]
fig, axes = plt.subplots(1, len(plot_cols), figsize=(4 * len(plot_cols), 3))
if len(plot_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, plot_cols):
    orders_df[col].hist(bins=40, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(col)
plt.suptitle("Numeric distributions (orders)")
plt.tight_layout()
plt.show()


In [ ]:
# Frequency tables for categorical columns in orders
cat_cols_orders = [c for c in orders_df.columns if c not in num_cols_orders]
for col in cat_cols_orders[:6]:
    print(f"\n{col} - value counts (top 10):")
    print(orders_df[col].value_counts(dropna=False).head(10))


In [ ]:
# Correlation matrix (numeric fields in orders) — matplotlib only
corr = orders_df[num_cols_orders].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Correlation matrix - orders (numeric)")
plt.tight_layout()
plt.show()


In [ ]:
# Relationship: order_total vs is_fraud (example)
plt.figure(figsize=(6, 4))
for label, g in orders_df.groupby("is_fraud"):
    g["order_total"].hist(bins=40, alpha=0.5, label=f"is_fraud={label}")
plt.xlabel("order_total")
plt.ylabel("count")
plt.legend()
plt.title("Order total by fraud label")
plt.tight_layout()
plt.show()


In [ ]:
conn.close()
print("EDA cells complete.")


### Schema summary (what we found in `shop.db`)

After inspection, typical contents are:

| Table | Role |
|-------|------|
| **orders** | One row per order; includes **`is_fraud`** (target) and checkout-time inputs. |
| **customers** | Demographics and segment; join on `customer_id`. |
| **order_items** | Line items per order; aggregate to order-level signals. |
| **products** | Product metadata; join via `order_items` for category diversity. |
| **product_reviews** | Could support *historical* review features before `order_datetime` (not used here to keep the first version simpler—add if needed). |
| **shipments** | Post-order; **not used** as features for checkout fraud (would be leakage of “future” process). |

**`orders` columns (candidates):** identifiers and keys (`order_id`, `customer_id`), timestamps (`order_datetime`), location/payment/device fields (`billing_zip`, `shipping_zip`, `shipping_state`, `payment_method`, `device_type`, `ip_country`), promo flags, monetary fields (`order_subtotal`, `shipping_fee`, `tax_amount`, `order_total`), and **`risk_score`** alongside **`is_fraud`**.

**Target:** `is_fraud` (binary).

**Features:** all safe checkout-time columns above **except** identifiers we drop from X, and **excluding `risk_score`** (documented in the schema as a label/risk signal—treating it as **leakage** relative to learning fraud from organic inputs).

If your database differs, re-run the inspection cells; the modeling section builds column lists **from whatever columns remain** after drops.


## 3. Data preparation

We build one modeling table by:

1. **Joining** `orders` → `customers` on `customer_id`.
2. **Aggregating** `order_items` (and optional category counts via `products`).
3. **Removing leakage:** drop `risk_score` (and never use `shipments` targets such as `late_delivery`).
4. **Parsing** `order_datetime` for time-based features (next section).

**Manual adjustment:** If `is_fraud` or `order_datetime` is missing in your DB, stop and fix the schema—the notebook asserts they exist.


In [ ]:
conn = sqlite3.connect(DB_PATH)

orders = pd.read_sql("SELECT * FROM orders", conn)
customers = pd.read_sql("SELECT * FROM customers", conn)
items = pd.read_sql("SELECT * FROM order_items", conn)
products = pd.read_sql("SELECT * FROM products", conn)

# Aggregates per order from line items
item_agg = (
    items.groupby("order_id")
    .agg(
        n_line_items=("order_item_id", "count"),
        total_units=("quantity", "sum"),
        max_line_total=("line_total", "max"),
        avg_unit_price=("unit_price", "mean"),
        line_total_sum=("line_total", "sum"),
    )
    .reset_index()
)

cat_agg = pd.read_sql(
    """
    SELECT oi.order_id,
           COUNT(DISTINCT p.category) AS n_distinct_categories
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY oi.order_id
    """,
    conn,
)

conn.close()

# Merge
df = orders.merge(customers, on="customer_id", how="left", suffixes=("", "_cust"))
df = df.merge(item_agg, on="order_id", how="left")
df = df.merge(cat_agg, on="order_id", how="left")

# Line-item aggregates: if an order had no rows (data issue), fill zeros
for c in ["n_line_items", "total_units", "max_line_total", "avg_unit_price", "line_total_sum", "n_distinct_categories"]:
    if c in df.columns:
        df[c] = df[c].fillna(0)

print("Modeling table shape after joins:", df.shape)
print("Columns:", list(df.columns))


In [ ]:
# --- Leakage / invalid feature guardrails ---
assert "is_fraud" in df.columns, "Target column is_fraud missing from orders."

LEAKAGE_DROP = []
if "risk_score" in df.columns:
    LEAKAGE_DROP.append("risk_score")
    print("Excluding risk_score (documented as risk/label signal - leakage for learning fraud from raw checkout features).")

# order_id is an identifier only (drop from X later / here to avoid treating as feature)
if "order_id" in df.columns:
    LEAKAGE_DROP.append("order_id")

df = df.drop(columns=[c for c in LEAKAGE_DROP if c in df.columns], errors="ignore")
# Keep customer_id until after feature engineering (needed for chronological history features).


In [ ]:
# Parse datetimes (used for feature engineering next)
if "order_datetime" not in df.columns:
    raise ValueError("order_datetime missing - cannot build time features.")

df["order_datetime"] = pd.to_datetime(df["order_datetime"], errors="coerce")
# Customer created_at if present
if "created_at" in df.columns:
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

print("order_datetime range:", df["order_datetime"].min(), "->", df["order_datetime"].max())
print("Missing order_datetime after parse:", df["order_datetime"].isna().sum())


## 4. Feature engineering

We only add signals that would be known **when the order is placed**:

| Feature idea | Why it might help |
|--------------|-------------------|
| **Time-of-day / day-of-week** | Fraud bots and stolen-card abuse often cluster at odd hours or weekends. |
| **Monetary ratios** (e.g. shipping vs subtotal, tax vs subtotal) | Unusual fee patterns can separate risky orders. |
| **Zip mismatch** | Billing vs shipping mismatch is a common fraud signal. |
| **Cart shape** (line count, units, category diversity) | Fraudulent orders sometimes look different from typical baskets. |
| **Customer tenure** (account age at order) | Brand-new accounts can be riskier. |
| **Prior order count / prior spend** (within this dataset, causal order) | Repeat behavior vs first-time spikes; computed with sort + groupby so only **past** orders count. |

If any engineered column is not available at prediction time in your real app, remove it.


In [ ]:
df = df.sort_values(["customer_id", "order_datetime"]).reset_index(drop=True)

# Time features from order timestamp
df["order_hour"] = df["order_datetime"].dt.hour
df["order_dow"] = df["order_datetime"].dt.dayofweek
df["order_month"] = df["order_datetime"].dt.month
df["is_weekend"] = df["order_dow"].isin([5, 6]).astype(int)

# Account age at order (days)
if "created_at" in df.columns:
    df["customer_tenure_days"] = (df["order_datetime"] - df["created_at"]).dt.days
    df["customer_tenure_days"] = df["customer_tenure_days"].clip(lower=0)
else:
    df["customer_tenure_days"] = np.nan

# Monetary ratios (avoid divide-by-zero)
eps = 1e-6
df["shipping_to_subtotal"] = df["shipping_fee"] / (df["order_subtotal"].abs() + eps)
df["tax_to_subtotal"] = df["tax_amount"] / (df["order_subtotal"].abs() + eps)
df["subtotal_to_total"] = df["order_subtotal"] / (df["order_total"].abs() + eps)

# Zip mismatch flag
if {"billing_zip", "shipping_zip"}.issubset(df.columns):
    df["zip_mismatch"] = (df["billing_zip"].fillna("") != df["shipping_zip"].fillna("")).astype(int)
else:
    df["zip_mismatch"] = 0

# Customer history (only prior orders in chronological order)
df["prior_order_count"] = df.groupby("customer_id").cumcount()
df["prior_total_spend"] = df.groupby("customer_id")["order_total"].cumsum() - df["order_total"]

# Line totals consistency check feature (data quality / weird carts)
if {"line_total_sum", "order_subtotal"}.issubset(df.columns):
    df["line_vs_subtotal_diff"] = df["line_total_sum"] - df["order_subtotal"]
else:
    df["line_vs_subtotal_diff"] = np.nan

print(df[[c for c in ["prior_order_count", "shipping_to_subtotal", "zip_mismatch", "n_line_items"] if c in df.columns]].describe())


In [ ]:
# Remove customer_id before defining X (not available as a generic categorical at scoring time for new customers —
# we already encoded history in prior_order_count / prior_total_spend).
if "customer_id" in df.columns:
    df = df.drop(columns=["customer_id"])
    print("Dropped customer_id after history features were computed.")


## 5. Modeling

**Train/test split:** stratified on `is_fraud` (class imbalance).

**Preprocessing:** `ColumnTransformer` + `Pipeline` so training is reproducible:

- **Numeric:** median imputation + `StandardScaler` (helps linear models; tree models are mostly unaffected).
- **Categorical:** constant imputation + `OneHotEncoder(handle_unknown="ignore")` with a cap on categories for high-cardinality text fields.

We then train several classifiers inside the same pipeline and compare them.


In [ ]:
TARGET = "is_fraud"
y = df[TARGET].astype(int)

feature_drop = [TARGET, "order_datetime"]
# Drop columns not useful as model inputs
for c in [
    "full_name",
    "email",
    "birthdate",
    "created_at",
    "promo_code",
    "sku",
    "product_name",
]:
    if c in df.columns:
        feature_drop.append(c)

X = df.drop(columns=[c for c in feature_drop if c in df.columns], errors="ignore")

# Split numeric vs categorical (avoid pandas stringdtype ambiguity)
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Numeric features (n=%d):" % len(numeric_features), numeric_features)
print("\nCategorical features (n=%d):" % len(categorical_features), categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)
print("\nTrain size:", X_train.shape, "Test size:", X_test.shape)


In [ ]:
# Fresh ColumnTransformer per pipeline (avoid sharing one fitted preprocessor across models)


def make_preprocessor(num_cols, cat_cols):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False, max_categories=25),
            ),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols),
        ]
    )


def make_clf_pipeline(classifier):
    return Pipeline(
        steps=[
            ("prep", make_preprocessor(numeric_features, categorical_features)),
            ("clf", classifier),
        ]
    )

models = {
    "logistic_regression": make_clf_pipeline(
        LogisticRegression(max_iter=5000, class_weight="balanced", solver="lbfgs")
    ),
    "random_forest": make_clf_pipeline(
        RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1,
        )
    ),
    "gradient_boosting": make_clf_pipeline(
        GradientBoostingClassifier(random_state=42)
    ),
    "hist_gradient_boosting": make_clf_pipeline(
        HistGradientBoostingClassifier(random_state=42, class_weight="balanced")
    ),
    "extra_trees": make_clf_pipeline(
        ExtraTreesClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
    ),
}

print("Pipelines ready:", list(models.keys()))


In [ ]:
# Same split for fair comparison + quick CV on training fold
cv_summary = []
for name, pipe in models.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
    cv_summary.append({"model": name, "cv_roc_auc_mean": scores.mean(), "cv_roc_auc_std": scores.std()})
cv_df = pd.DataFrame(cv_summary).sort_values("cv_roc_auc_mean", ascending=False)
print("5-fold CV ROC AUC (train fold only):")
print(cv_df.to_string(index=False))


In [ ]:
# Fit all on full training set for test-set evaluation below
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    print(f"Fitted {name}")


## 6. Evaluation

We report **accuracy, precision, recall, F1, ROC AUC**, plus **confusion matrices** and **classification reports**.

**Which metric matters most for fraud?**  
Missed fraud (false negatives) often costs more than reviewing extra orders (false positives), so **recall** is frequently prioritized—but too many false positives hurt revenue and trust. **F1** balances both; **ROC AUC** helps compare models before choosing a threshold.

We then **tune** the strongest candidate with `RandomizedSearchCV`, discuss **thresholds**, and analyze **feature importance** (permutation importance on the held-out logic).


In [ ]:
def evaluate_model(name, pipe, X_te, y_te, threshold=0.5):
    proba = pipe.predict_proba(X_te)[:, 1]
    y_pred = (proba >= threshold).astype(int)
    return {
        "model": name,
        "accuracy": accuracy_score(y_te, y_pred),
        "precision": precision_score(y_te, y_pred, zero_division=0),
        "recall": recall_score(y_te, y_pred, zero_division=0),
        "f1": f1_score(y_te, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_te, proba),
    }

rows = []
for name, pipe in models.items():
    rows.append(evaluate_model(name, pipe, X_test, y_test))
results_df = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
print("Test-set metrics (default threshold 0.5):")
print(results_df.to_string(index=False))


In [ ]:
best_name = results_df.iloc[0]["model"]
best_pipe = models[best_name]
print("Best model by ROC AUC on test:", best_name)

ConfusionMatrixDisplay.from_estimator(best_pipe, X_test, y_test, cmap="Blues")
plt.title(f"Confusion matrix - {best_name} (threshold=0.5)")
plt.show()

y_proba = best_pipe.predict_proba(X_test)[:, 1]
y_pred = best_pipe.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))


In [ ]:
# ROC curve for best model
fpr, tpr, thr = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"{best_name} (AUC={roc_auc_score(y_test, y_proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="random")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve (test set)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Threshold discussion: trade precision vs recall
thresholds = np.linspace(0.05, 0.95, 19)
tr_rows = []
for t in thresholds:
    y_hat = (y_proba >= t).astype(int)
    tr_rows.append(
        {
            "threshold": t,
            "precision": precision_score(y_test, y_hat, zero_division=0),
            "recall": recall_score(y_test, y_hat, zero_division=0),
            "f1": f1_score(y_test, y_hat, zero_division=0),
        }
    )
thr_df = pd.DataFrame(tr_rows)
print(thr_df.to_string(index=False))

# Example: pick threshold that maximizes F1 on test (for demo only — in practice use validation/CV)
best_t = thr_df.loc[thr_df["f1"].idxmax(), "threshold"]
print(f"\nExample: threshold with max test F1 ~ {best_t:.2f} (illustrative; tune on validation in real projects).")


In [ ]:
# Hyperparameter tuning on the same estimator family as the test-set ROC AUC winner
base_clf = models[best_name].named_steps["clf"]
search = None

if isinstance(base_clf, HistGradientBoostingClassifier):
    param_distributions = {
        "clf__max_depth": [None, 8, 12, 16],
        "clf__max_leaf_nodes": [31, 63, 127, None],
        "clf__learning_rate": np.logspace(-2, 0, 6),
        "clf__min_samples_leaf": [5, 10, 20, 40],
        "clf__l2_regularization": np.logspace(-4, 1, 6),
    }
    est = HistGradientBoostingClassifier(random_state=42, class_weight="balanced")
elif isinstance(base_clf, RandomForestClassifier):
    param_distributions = {
        "clf__n_estimators": [100, 200, 400],
        "clf__max_depth": [None, 8, 16, 24],
        "clf__min_samples_leaf": [1, 2, 4, 8],
    }
    est = RandomForestClassifier(
        random_state=42, class_weight="balanced_subsample", n_jobs=-1
    )
elif isinstance(base_clf, ExtraTreesClassifier):
    param_distributions = {
        "clf__n_estimators": [200, 400, 600],
        "clf__max_depth": [None, 12, 20],
        "clf__min_samples_leaf": [1, 2, 4],
    }
    est = ExtraTreesClassifier(random_state=42, class_weight="balanced", n_jobs=-1)
elif isinstance(base_clf, LogisticRegression):
    param_distributions = {"clf__C": np.logspace(-2, 2, 12)}
    est = LogisticRegression(max_iter=5000, class_weight="balanced", solver="lbfgs")
elif isinstance(base_clf, GradientBoostingClassifier):
    param_distributions = {
        "clf__n_estimators": [100, 200],
        "clf__learning_rate": [0.03, 0.05, 0.1, 0.2],
        "clf__max_depth": [2, 3, 4],
    }
    est = GradientBoostingClassifier(random_state=42)
else:
    est = None

if est is not None:
    search = RandomizedSearchCV(
        make_clf_pipeline(est),
        param_distributions=param_distributions,
        n_iter=22,
        scoring="f1",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )
    search.fit(X_train, y_train)
    best_tuned = search.best_estimator_
    print("Tuned model:", type(est).__name__)
    print("Best params:", search.best_params_)
    print("Best CV F1:", search.best_score_)
    print(evaluate_model(best_name + "_tuned", best_tuned, X_test, y_test))
else:
    best_tuned = None
    print("No tuning grid defined for this estimator; export the baseline winner.")


In [ ]:
# Final model: use tuned pipeline only if it improves test F1 vs the same baseline family
base_f1 = f1_score(y_test, best_pipe.predict(X_test))
if best_tuned is not None:
    tuned_f1 = f1_score(y_test, best_tuned.predict(X_test))
    final_model = best_tuned if tuned_f1 >= base_f1 else best_pipe
    final_label = (best_name + "_tuned") if final_model is best_tuned else best_name
    print("Test F1 baseline:", base_f1, "| tuned:", tuned_f1)
else:
    final_model = best_pipe
    final_label = best_name
print("Chosen for export:", final_label)


In [ ]:
# Feature importance: permutation importance on the exported pipeline
# Note: sklearn permutes raw input columns (here: same columns as X_train), not one-hot expanded columns.
result = permutation_importance(
    final_model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
    scoring="roc_auc",
)
feat_names_raw = list(X_train.columns)
imp = pd.Series(result.importances_mean, index=feat_names_raw).sort_values(ascending=False).head(25)
print("Top permutation importances (ROC AUC drop; raw features):")
print(imp)

imp.plot(kind="barh", figsize=(8, 8), color="teal")
plt.gca().invert_yaxis()
plt.title("Permutation importance (top features)")
plt.tight_layout()
plt.show()


In [ ]:
# Univariate screening on transformed training matrix (SelectKBest - illustrative)
X_tr_t = final_model.named_steps["prep"].transform(X_train)
kbest = SelectKBest(score_func=f_classif, k=min(20, X_tr_t.shape[1]))
kbest.fit(X_tr_t, y_train)
feat_names_t = final_model.named_steps["prep"].get_feature_names_out()
scores = pd.Series(kbest.scores_, index=feat_names_t).sort_values(ascending=False).head(20)
print("Top F-scores (SelectKBest, transformed features):")
print(scores)


## 7. Deployment

We serialize the **full sklearn `Pipeline`** (preprocessing + classifier) with **joblib** so scoring new orders uses the same imputation, scaling, and encoding.

**Integration sketch:** when an order is submitted, the web tier loads the pipeline once, builds a single-row DataFrame with the same columns as training, calls `predict_proba` for **fraud probability**, then applies a **business threshold** for the final **block/review/allow** decision.


In [ ]:
MODEL_DIR = DB_PATH.parent / "models"
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "fraud_model.joblib"

joblib.dump(final_model, MODEL_PATH)
print("Saved:", MODEL_PATH.resolve())


In [ ]:
# Load and score (deployment-style)
loaded = joblib.load(MODEL_PATH)

# Example: take one holdout row as "new" data
sample_X = X_test.iloc[[0]].copy()
p_fraud = float(loaded.predict_proba(sample_X)[0, 1])
pred = int(loaded.predict(sample_X)[0])
print("Example fraud probability:", round(p_fraud, 4))
print("Predicted class (default 0.5 threshold):", pred)

# Small batch
probs = loaded.predict_proba(X_test.iloc[:5])[:, 1]
print("First 5 probabilities:", np.round(probs, 4))


## 8. Final summary

Fill in after a full run (values depend on your data split):

- **Best model:** Usually tuned **HistGradientBoosting** or the top baseline from `results_df`.
- **Best metrics:** Compare `results_df` and tuned test scores; fraud projects often emphasize **recall** at an acceptable **precision**.
- **Key features:** See permutation importance and `SelectKBest` rankings (payment channel, geo, cart shape, tenure, ratios, etc.).
- **Next steps:** monitor drift, refresh labels, add **cost-sensitive** thresholds, collect investigator feedback, and consider **real-time** feature stores for production-scale joins.

---

**Files:** trained artifact at `models/fraud_model.joblib`. Optional batch trainer: `train_fraud_model.py`.
